# 06 — Сравнение трёх теорий: `classic` / `karman` / `ktn`

Лестница моделей одним ключом `[model] theory` (честные термины):

* **`classic`** — линейная теория Кирхгофа (расщепление бигармоники);
* **`karman`** — геометрически-**НЕЛИНЕЙНОЕ** решение Фёппля–Кармана: прогиб
  входит квадратично в мембранные деформации, поле усилий `N` ужестчает
  пластину (мембранная связь);
* **`ktn`** — **ЛИНЕЙНЫЕ** поправки поперечного сдвига/обжатия постобработкой
  на решении Кирхгофа (не нелинейная теория).

Каноническая пластина — защемлённый круг, **неподвижная** кромка
(`inplane_bc = immovable`), равномерная нагрузка. Безразмерно: `E = h = a = 1`,
`ν = 0.3` ⇒ безразмерная нагрузка `P̄ = q0`, прогиб `w/h = w_max`.

Эталоны большого прогиба (разной математической природы) берутся из
`plate_solver.benchmarks`; case-файлы:
`cases/ci/karman_circle_clamped_immovable.toml`,
`cases/ci/karman_square_clamped_immovable.toml`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from plate_solver import benchmarks as bm
from plate_solver.config import Config
from plate_solver.geometry import make_circle
from plate_solver.membrane import KarmanPlate

nu = 0.3
dom = make_circle(1.0)


def solve_karman(P_bar, Q=120, bc="clamped", inplane="immovable"):
    '''Решить Карман на круге при безразмерной нагрузке P̄ (= q0).'''
    cfg = Config(E=1.0, h=1.0, nu=nu, a=1.0, q0=float(P_bar), p=12, Q=Q,
                 n_load_steps=max(2, int(round(P_bar / 2))),
                 karman_tol=1e-6, karman_max_iter=300)
    kp = KarmanPlate.from_config(dom, cfg, bc_type=bc, inplane_bc=inplane)
    return kp, kp.solve_uniform()

## 1. Свип нагрузки: три теории на одной пластине

Для каждого уровня `P̄` берём нелинейный `w_karman` и линейный `w_classic`
(= `KarmanResult.w_max_classic`, решение Кирхгофа при `N = 0`). Под РАВНОМЕРНОЙ
нагрузкой срединный прогиб `ktn` БЛИЗОК к `classic` (`Δq = 0`; расхождение
порядка `(h/L)²`): поправка сдвига/обжатия живёт в лицевых величинах — п. 4.

In [ ]:
p_bars = np.array([0.5, 1.0, 2.0, 3.196, 4.561, 6.321, 8.635, 11.71])
w_karman, w_classic, converged = [], [], []
for pb in p_bars:
    _, r = solve_karman(pb)
    w_karman.append(r.w_max)
    w_classic.append(r.w_max_classic)
    converged.append(r.converged)
w_karman = np.array(w_karman)
w_classic = np.array(w_classic)
print("сошлось на всех уровнях:", all(converged))
for pb, wk, wc in zip(p_bars, w_karman, w_classic, strict=True):
    print(f"  P̄={pb:6.3f}:  w_karman/h={wk:.4f}  w_classic/h={wc:.4f}  "
          f"ужесточение={wc/wk:.2f}x")

## 2. График «нагрузка–прогиб» с наложенными эталонами

`classic` — прямая (линейная жёсткость); `karman` — загиб ВНИЗ (мембранное
ужесточение), ложится на ряды **Way**; при большой нагрузке приближается СНИЗУ
к асимптоте мембраны **Hencky** `0.653·P̄^{1/3}`. `ktn` (срединный) под
равномерной нагрузкой близок к `classic` (расхождение порядка `(h/L)²`).

In [ ]:
way = bm.way_clamped_circle()
pb_fine = np.linspace(0.05, p_bars.max(), 200)
hencky = np.array([bm.hencky_center_deflection(x, nu) for x in pb_fine])

fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.plot(p_bars, w_classic, "o--", color="tab:blue", label="classic (Кирхгоф)")
ax.plot(p_bars, w_karman, "s-", color="tab:red", label="karman (нелинейный)")
ax.plot(p_bars, w_classic, ".", ms=12, color="gray",
        label="ktn (срединный ≈ classic, Δq=0)")
ax.plot(way[:, 0], way[:, 1], "k*", ms=13, label="эталон Way (ряды)")
ax.plot(pb_fine, hencky, ":", color="crimson",
        label="асимптота Hencky 0.653·P̄^{1/3}")
ax.set_xlabel("P̄ = p·a⁴/(E·h⁴)")
ax.set_ylabel("w / h")
ax.set_title("Нагрузка–прогиб: три теории и независимые эталоны")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

## 3. Величина эффекта: отношение `w_karman / w_classic`

Отношение убывает с нагрузкой — мембранное ужесточение растёт с прогибом
(при `w/h → 0` теории совпадают, отношение → 1).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_bars, w_karman / w_classic, "o-", color="tab:red")
ax.axhline(1.0, ls=":", color="gray")
ax.set_xlabel("P̄")
ax.set_ylabel("w_karman / w_classic")
ax.set_title("Мембранное ужесточение члена Кармана")
ax.grid(alpha=0.3)
plt.show()

## 4. Что «даёт» член Кармана: поле мембранного усилия `N`

Мембранное поле — то, чего НЕТ в линейной теории. Показываем среднее усилие
`½(N_x+N_y)` при умеренно большом прогибе (`P̄ = 6.321`, `w/h ≈ 0.8`):
неподвижная кромка держит растяжение (`N > 0`).

In [ ]:
kp, r = solve_karman(6.321)
g = 121
Xg, Yg = np.meshgrid(np.linspace(-1, 1, g), np.linspace(-1, 1, g))
inside = dom.omega(Xg, Yg) > 0.0
Nx = np.full(Xg.shape, np.nan)
Ny = np.full(Xg.shape, np.nan)
nx, ny, nxy = kp.membrane_forces_at(r.cu, r.cv, r.cw, Xg[inside], Yg[inside])
Nx[inside], Ny[inside] = nx, ny
Nmean = 0.5 * (Nx + Ny)

fig, ax = plt.subplots(figsize=(5.6, 5))
cs = ax.contourf(Xg, Yg, Nmean, levels=20, cmap="viridis")
fig.colorbar(cs, label="½(N_x + N_y)   (растяжение > 0)")
ax.set_aspect("equal")
ax.set_title("Мембранное усилие (P̄ = 6.321): вклад теории Кармана")
plt.show()

## 5. Подпись КТН: где виден сдвиг/обжатие

Под РАВНОМЕРНОЙ нагрузкой срединный прогиб `ktn` БЛИЗОК к `classic` (расхождение
порядка `(h/L)²`, `Δq = 0`), а поправка поперечного сдвига/обжатия проявляется в
ЛИЦЕВЫХ величинах: смещение контактирующей (нижней) лицевой `dh = w_bot − w`
(NOTES §21). Толщина `h = 0.1` (физичный линейный режим, `w/h ≪ 1`) — относительная
поправка `dh / w ~ (h/L)²`.

In [ ]:
from plate_solver.dispatch import solve
from plate_solver.problem import Problem

prob = Problem.from_dict({
    "geometry": {"kind": "circle", "a": 1.0},
    "bc": {"type": "clamped"},
    "load": {"type": "uniform", "q0": 4.0},
    "model": {"theory": "ktn", "E": 2.1e6, "nu": 0.3, "h": 0.1},
    "discretization": {"p": 10, "Q": 140, "grid_n": 60},
})
res = solve(prob)
w_top, w_bot, dh = res.faces_on_grid()
rel_mid = abs(res.w_max - res.w_max_classic) / res.w_max_classic
print(f"w_max (КТН, срединный) = {res.w_max:.6e}   (w/h = {res.w_max/0.1:.4f}, линейный режим)")
print(f"w_max_classic          = {res.w_max_classic:.6e}")
print(f"срединное расхождение КТН↔classic = {rel_mid*100:.3f}%  (порядка (h/L)²)")
print(f"относит. смещение лицевой max|dh|/w = {np.nanmax(np.abs(dh))/res.w_max:.4f}"
      "  (эффект КТН — в лицевых)")

fig, ax = plt.subplots(figsize=(5.6, 5))
cs = ax.contourf(res.Xg, res.Yg, dh, levels=20, cmap="coolwarm")
fig.colorbar(cs, label="dh = w_bot − w_срединный")
ax.set_aspect("equal")
ax.set_title("КТН: смещение лицевой поверхности (сдвиг/обжатие)")
plt.show()

## 6. Итог: какой доп. член у каждой теории и где виден

| Теория | Дополнительный член | Где виден | Природа |
|---|---|---|---|
| `classic` | — | линейная жёсткость | линейный Кирхгоф |
| `karman` | мембранная связь `N:∇∇w` (квадрат по `w`) | загиб «нагрузка–прогиб» вниз; поле `N` | геометрическая нелинейность |
| `ktn` | сдвиг `h_Ψ²` + обжатие `h_*²` | ЛИЦЕВЫЕ прогиб/напряжения (`dh`); под равномерной нагрузкой срединный `w ≈ classic` (расхождение `(h/L)²`) | линейные поправки `(h/L)²` |

**Оговорки.**

* **Неподвижная кромка обязательна** для эффекта Кармана: `movable` (втягивание
  кромки) почти снимает мембранное натяжение (§3.3 постановки).
* **`Δq = 0`**: у `ktn` под равномерной нагрузкой поправка почти не смещает
  срединную поверхность (расхождение `(h/L)²`) — она живёт в лицевых величинах.
* **Разные `ν` эталонов** (риск №1): `ν = 0.30` у Way/Тимошенко/Hencky,
  `ν = 0.316` у Levy — не смешивать в одном допуске (приложение B, `benchmarks`).
* Полная нелинейная КТН (`ktn_full` = Карман + сдвиг + обжатие) — направление
  развития (релиз v0.5.0).